# 08a · Attention from Scratch `[local]` (benchmark: also run on Kaggle)

**Goal:** understand attention well enough to write it from memory. You'll build scaled dot-product attention, multi-head attention, positional encodings (learned, sinusoidal, RoPE), FlashAttention's memory trick, windowed attention and a channel-attention block for CNNs, checking each against PyTorch, then benchmark them.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Attention as a soft dictionary lookup; self vs. cross; heads; why positions; why it's slow |
| 📐 Math | Scaled dot-product, the √d, masks, multi-head, sinusoidal and rotary positions, online softmax |
| 🔨 From scratch | Attention, multi-head attention, RoPE, FlashAttention-style chunking, window attention, SE block |
| 🚀 Benchmark | Your attention vs. the chunked version vs. PyTorch's fused kernel: speed and memory |
| 📊 Evaluate | Numerical agreement in float32 and float16 |
| 🧪 Experiment | Do positions matter? Does RoPE handle longer inputs than it trained on? Does SE help a CNN? |

The code you write here also lives in `cvlib/transformer.py`, which 08b uses to build a full Vision Transformer.

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import os, math, time, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from cvlib.datasets import cifar10
from cvlib.loaders import GPULoader, random_crop_flip, CIFAR_MEAN, CIFAR_STD
from cvlib.models import conv_bn
from cvlib import train as cvtrain

device = env.device
FULL = env.platform == "kaggle" or os.environ.get("CV_FULL") == "1"

def sync():
    if device == "cuda": torch.cuda.synchronize()
    elif device == "mps": torch.mps.synchronize()

---
# 🧠 1 · Intuition

### Attention = a soft dictionary lookup
A Python dict lookup finds the **one** key that exactly matches your query and returns its value. Attention does a *soft* version:
1. compare your **query** with **every key** → a similarity score for each
2. turn the scores into weights that sum to 1 (softmax)
3. return the **weighted average of the values**

So instead of "the value for key X" you get "mostly the values whose keys look like what I'm asking for".

In a transformer, every token (a word, or an image patch) produces all three, using three learned linear layers:
- **query:** "what am I looking for?"
- **key:** "what do I contain?" (used for matching)
- **value:** "what do I hand over if you pick me?"

A patch showing a dog's ear can ask "where's the rest of this animal?" and pull in information from the dog's body patches, **wherever they are in the image**. A conv layer can only see its 3×3 neighbours; attention sees everything from layer 1.

### Self-attention vs. cross-attention
- **Self-attention:** queries, keys and values all come from the same sequence. Patches talk to each other (ViT, 08b).
- **Cross-attention:** queries come from one sequence, keys and values from another. DETR's object "slots" query the image (module 09); a VLM's text queries image features (module 13).

### Multi-head
One attention averages everything into a single blend. **Multi-head** attention runs several smaller attentions in parallel (e.g. 8 heads of 64 dims instead of 1 of 512), so one head can track "same object", another "what's to my left", another "same color". Their outputs are concatenated and mixed by a final linear layer. Same parameter count, more kinds of relationships.

### Attention doesn't know about order
Attention compares contents only. Shuffle the input tokens and you get the same outputs, shuffled the same way. For images that's fatal: "sky above grass" and "grass above sky" would look identical. So we **add position information**:
- **learned:** one trainable vector per position (ViT)
- **sinusoidal:** fixed sine/cosine patterns (the original Transformer)
- **RoPE (rotary):** rotate queries and keys by an angle that depends on position, so their match depends on *how far apart* they are (most modern LLMs and VLMs)

### Why attention is slow on long inputs
Every token compares with every other token: $N$ tokens → an $N \times N$ score matrix. Double the tokens and the cost goes up **4×**.

| Input | Tokens | Score matrix (one head, float32) |
|---|---|---|
| 224×224 image, 16×16 patches (ViT) | 196 | 0.15 MB |
| 1024×1024 image, 16×16 patches | 4,096 | 64 MB |
| 224×224 image, one token per pixel | 50,176 | **10 GB** |

That's why ViTs use patches, and why high-resolution vision (detection, documents) needs tricks:
- **FlashAttention** computes *exactly* the same result without ever storing the full $N \times N$ matrix: it processes keys in blocks and keeps a running softmax in fast on-chip memory. Same math, far less memory, faster.
- **Windowed attention** (Swin, 08b) only lets tokens attend inside small local windows: much less compute, but it's an approximation, so windows are shifted between layers to let information spread.

### Attention inside CNNs
The idea "compute weights from the input, then reweight" predates transformers. **Squeeze-and-Excitation (SE)** blocks look at a whole feature map, decide which **channels** matter for *this* image, and scale them up or down. They're in EfficientNet and MobileNetV3 (module 04).

---
# 📐 2 · The math

### Scaled dot-product attention
Stack the queries as rows of $Q$ ($N_q \times d$), and keys and values as $K, V$ ($N_k \times d$):

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d}}\right) V$$

- $QK^\top$: every query dotted with every key → an $N_q \times N_k$ score matrix
- softmax over each **row**: each query's weights sum to 1
- multiply by $V$: each output row is a weighted average of value rows

**Why divide by $\sqrt{d}$?** If $q$ and $k$ have $d$ independent entries with variance 1, their dot product has variance $d$. With $d = 64$, scores have a standard deviation of 8, the softmax becomes nearly one-hot, and its gradients vanish. Dividing by $\sqrt{d}$ brings the variance back to 1.

**Masks** set some scores to $-\infty$ before the softmax, giving those positions exactly zero weight: padding tokens, or future tokens in a text decoder (a *causal* mask).

### Multi-head attention
With $h$ heads and model width $D$, each head works in $d = D/h$ dimensions:

$$\text{head}_i = \text{Attention}(X W_i^Q,\; X W_i^K,\; X W_i^V) \qquad \text{MHA}(X) = [\text{head}_1, \dots, \text{head}_h]\, W^O$$

In code, all heads' projections are one big $D \times 3D$ matrix (the fused **qkv** layer), and the output is reshaped into heads.

### Positional encodings
**Sinusoidal:** position $p$, dimension pairs $2i, 2i{+}1$:
$$PE_{p,2i} = \sin(p / 10000^{2i/D}) \qquad PE_{p,2i+1} = \cos(p / 10000^{2i/D})$$
Fast-changing waves for nearby positions, slow ones for far positions, like the hands of a clock.

**RoPE:** treat each pair of query/key dimensions as a 2-D point and rotate it by angle $p\,\theta_i$ at position $p$. Rotating $q$ by angle $m\theta$ and $k$ by $n\theta$ gives

$$\langle R_{m\theta}\, q,\; R_{n\theta}\, k\rangle = \langle q,\; R_{(n-m)\theta}\, k\rangle$$

so the score depends only on the **relative** offset $n - m$, never on absolute positions. Nothing is added to the token itself.

### Cost
For $N$ tokens of width $D$: the projections cost $O(N D^2)$, the scores and weighted sum $O(N^2 D)$, and storing the scores needs $O(N^2)$ memory per head. Beyond $N \approx D$ the quadratic part dominates.

### Online softmax (the core of FlashAttention)
Softmax needs a sum over the whole row, which seems to require all scores at once. It doesn't: process keys in blocks and keep, for each query, the running max $m$, running denominator $\ell$ and running unnormalized output $o$. For a new block of scores $s$:

$$m' = \max(m, \max s) \qquad \ell' = \ell\, e^{m - m'} + \textstyle\sum e^{s - m'} \qquad o' = o\, e^{m - m'} + e^{s - m'}\, V_{\text{block}}$$

At the end, output $= o / \ell$. **Exact**, and memory is $O(N \times \text{block})$ instead of $O(N^2)$. (Subtracting the max is the usual trick to avoid overflow in `exp`.)

---
# 🔨 3 · From scratch

### 3.1 · Scaled dot-product attention, checked against PyTorch

In [ ]:
def attention(q, k, v, mask=None):
    """q: (..., Nq, d), k and v: (..., Nk, d). mask: bool, True = may attend. Returns (output, weights)."""
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])     # (..., Nq, Nk)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = scores.softmax(dim=-1)                               # each row sums to 1
    return weights @ v, weights

torch.manual_seed(0)
q, k, v = torch.randn(2, 4, 10, 16), torch.randn(2, 4, 12, 16), torch.randn(2, 4, 12, 16)   # (batch, heads, tokens, dim)
out, w = attention(q, k, v)
ref = F.scaled_dot_product_attention(q, k, v)
print("output shape:", tuple(out.shape), "| weights shape:", tuple(w.shape))
print(f"max |mine − PyTorch| = {(out - ref).abs().max():.2e}")
print("each row of weights sums to 1:", torch.allclose(w.sum(-1), torch.ones(1)))

causal = torch.ones(10, 10, dtype=torch.bool).tril()               # token i may only see tokens 0..i
out_c, w_c = attention(q[..., :10, :], k[..., :10, :], v[..., :10, :], causal)
ref_c = F.scaled_dot_product_attention(q[..., :10, :], k[..., :10, :], v[..., :10, :], is_causal=True)
print(f"causal mask: max diff {(out_c - ref_c).abs().max():.2e}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(w[0, 0].numpy(), cmap="viridis"); axes[0].set_title("weights: 10 queries × 12 keys")
axes[1].imshow(w_c[0, 0].numpy(), cmap="viridis"); axes[1].set_title("causal mask: no peeking ahead")
for a in axes: a.set_xlabel("key"); a.set_ylabel("query")
plt.tight_layout(); plt.show()

### 3.2 · Why divide by √d?
Random queries and keys of growing dimension $d$: how peaked does the softmax get with and without the scaling? (Max weight 1.0 = all attention on one key, so gradients to the other keys vanish.)

In [ ]:
dims = [4, 16, 64, 256, 1024]
rows = []
for d in dims:
    qq, kk = torch.randn(2000, d), torch.randn(64, d)
    s = qq @ kk.T
    for name, scale in [("no scaling", 1.0), ("÷ √d", 1 / math.sqrt(d))]:
        p = (s * scale).softmax(-1)
        rows.append(dict(d=d, version=name, score_std=(s * scale).std().item(), mean_max_weight=p.max(-1).values.mean().item()))
df = pd.DataFrame(rows)
print(df.pivot(index="d", columns="version", values=["score_std", "mean_max_weight"]).round(3))

Without scaling, the score spread grows like √d and the softmax quickly becomes "pick one key, ignore the rest". With it, the spread stays at 1 for every width.

### 3.3 · Multi-head attention
The version in `cvlib/transformer.py` (shown below) does self-attention and cross-attention with the same weights. Its fused **qkv** layer has the same layout as PyTorch's `nn.MultiheadAttention`, so we can copy weights across and compare outputs exactly.

In [ ]:
from cvlib.transformer import MultiHeadAttention
print(inspect.getsource(MultiHeadAttention.forward))

mha = MultiHeadAttention(dim=64, num_heads=4)
ref = nn.MultiheadAttention(64, 4, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(mha.qkv.weight); ref.in_proj_bias.copy_(mha.qkv.bias)
    ref.out_proj.weight.copy_(mha.proj.weight); ref.out_proj.bias.copy_(mha.proj.bias)

x, ctx = torch.randn(2, 10, 64), torch.randn(2, 30, 64)
with torch.no_grad():
    print(f"self-attention : max diff {(mha(x) - ref(x, x, x)[0]).abs().max():.2e}")
    print(f"cross-attention: max diff {(mha(x, ctx) - ref(x, ctx, ctx)[0]).abs().max():.2e}  (10 queries read 30 context tokens)")
print(f"parameters: {sum(p.numel() for p in mha.parameters()):,} = qkv 64×192+192 + proj 64×64+64")

### 3.4 · Attention ignores order: permutation equivariance
Shuffle the tokens. Without position information the output is just the old output, shuffled the same way.

In [ ]:
perm = torch.randperm(10)
with torch.no_grad():
    y, y_perm = mha(x), mha(x[:, perm])
    print("shuffle in → same shuffle out:", torch.allclose(y[:, perm], y_perm, atol=1e-6))
    pos = torch.randn(1, 10, 64)                       # add a position vector to each slot
    y, y_perm = mha(x + pos), mha(x[:, perm] + pos)
    print("with positions added:         ", torch.allclose(y[:, perm], y_perm, atol=1e-6))

### 3.5 · Sinusoidal positions and RoPE

In [ ]:
def sinusoidal(n_pos, dim):
    pos = torch.arange(n_pos)[:, None]
    freq = 1.0 / 10000 ** (torch.arange(0, dim, 2) / dim)
    pe = torch.zeros(n_pos, dim)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * freq), torch.cos(pos * freq)
    return pe

def rope(x, base=10000.0, offset=0):
    """Rotate each pair of dims (0,1), (2,3), ... of x (..., N, d) by angle position × θ_i."""
    N, d = x.shape[-2:]
    theta = 1.0 / base ** (torch.arange(0, d, 2, device=x.device) / d)        # one frequency per pair
    ang = (torch.arange(N, device=x.device) + offset)[:, None] * theta[None]   # (N, d/2)
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

pe = sinusoidal(64, 128)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].imshow(pe.T, aspect="auto", cmap="RdBu"); axes[0].set_xlabel("position"); axes[0].set_ylabel("dimension")
axes[0].set_title("sinusoidal encodings: fast waves at top, slow at bottom")
axes[1].imshow(pe @ pe.T, cmap="viridis"); axes[1].set_title("similarity between positions (peaks on the diagonal)")
plt.tight_layout(); plt.show()

# RoPE: the score between a query at position m and a key at position n depends only on n − m
qv, kv = torch.randn(1, 64), torch.randn(1, 64)
def rope_score(m, n):
    big = torch.zeros(max(m, n) + 1, 64)
    qs, ks = big.clone(), big.clone()
    qs[m], ks[n] = qv[0], kv[0]
    return (rope(qs)[m] @ rope(ks)[n]).item()
for m, n in [(0, 3), (10, 13), (50, 53), (0, 20), (30, 50)]:
    print(f"query at {m:>2}, key at {n:>2} (offset {n - m:>2}): score {rope_score(m, n):+.4f}")

Same offset → same score, wherever the pair sits. That's why RoPE generalizes well to new positions (tested in the 🧪 experiments).

### 3.6 · FlashAttention's trick: chunked attention with an online softmax
The real FlashAttention is a GPU kernel that also tiles the queries and keeps everything in on-chip memory. This version shows the algorithm: keys and values are processed in blocks, so the full $N \times N$ score matrix never exists.

In [ ]:
def chunked_attention(q, k, v, block=256):
    scale = 1 / math.sqrt(q.shape[-1])
    m = torch.full(q.shape[:-1] + (1,), float("-inf"), device=q.device, dtype=q.dtype)   # running max per query
    l = torch.zeros_like(m)                                                              # running softmax denominator
    o = torch.zeros_like(q)                                                              # running weighted sum of values
    for s in range(0, k.shape[-2], block):
        sc = q @ k[..., s:s + block, :].transpose(-2, -1) * scale                         # (..., Nq, block) only
        m_new = torch.maximum(m, sc.amax(-1, keepdim=True))
        p = torch.exp(sc - m_new)
        correction = torch.exp(m - m_new)                                                # rescale what we had so far
        l = l * correction + p.sum(-1, keepdim=True)
        o = o * correction + p @ v[..., s:s + block, :]
        m = m_new
    return o / l

q, k, v = torch.randn(1, 4, 1000, 64), torch.randn(1, 4, 1000, 64), torch.randn(1, 4, 1000, 64)
print(f"chunked vs. full attention: max diff {(chunked_attention(q, k, v, block=128) - attention(q, k, v)[0]).abs().max():.2e}")

### 3.7 · Windowed attention (Swin's approach)
Tokens on a 2-D grid attend only within non-overlapping $w \times w$ windows. On alternate layers the grid is **shifted** by $w/2$ (with `torch.roll`) so neighbouring windows exchange information. Let's see which tokens one query can see, and the cost.

In [ ]:
def window_partition(x, w):        # (B, H, W, C) → (B·nWindows, w·w, C)
    B, H, W, C = x.shape
    x = x.view(B, H // w, w, W // w, w, C).permute(0, 1, 3, 2, 4, 5)
    return x.reshape(-1, w * w, C)

def window_unpartition(x, w, B, H, W):
    C = x.shape[-1]
    x = x.view(B, H // w, W // w, w, w, C).permute(0, 1, 3, 2, 4, 5)
    return x.reshape(B, H, W, C)

def window_attention(x, w, shift=0):
    """Self-attention within w×w windows of a token grid x (B, H, W, C). (Real Swin also masks
    tokens that the roll wrapped around from the opposite edge; skipped here for clarity.)"""
    B, H, W, C = x.shape
    if shift: x = torch.roll(x, (-shift, -shift), dims=(1, 2))
    win = window_partition(x, w)
    out, _ = attention(win, win, win)
    out = window_unpartition(out, w, B, H, W)
    if shift: out = torch.roll(out, (shift, shift), dims=(1, 2))
    return out

# which tokens can query (row 5, col 5) see? Mark the query's token, run attention, look at its weights
H = W = 14; w = 7
def visible_from(r, c, shift):
    ids = torch.arange(H * W, dtype=torch.float32).view(1, H, W, 1)
    rolled = torch.roll(ids, (-shift, -shift), dims=(1, 2)) if shift else ids
    win = window_partition(rolled, w)[:, :, 0]                     # token ids in each window
    which = (win == r * W + c).any(1).nonzero()[0, 0]
    vis = torch.zeros(H * W); vis[win[which].long()] = 1
    return vis.view(H, W)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, (title, img) in zip(axes, [("global: everything", torch.ones(H, W)),
                                   ("window 7×7", visible_from(5, 5, 0)),
                                   ("shifted window", visible_from(5, 5, w // 2))]):
    ax.imshow(img, cmap="Blues", vmin=0, vmax=1.3); ax.plot(5, 5, "r*", ms=12); ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("tokens visible to the ★ query on a 14×14 grid"); plt.tight_layout(); plt.show()

xg = torch.randn(1, 56, 56, 64)                                    # Swin-T's first stage: 56×56 tokens
print(f"window output shape {tuple(window_attention(xg, 7).shape)}, shifted {tuple(window_attention(xg, 7, shift=3).shape)}")
n = 56 * 56
print(f"scores computed: global {n * n / 1e6:.1f}M vs. windowed {(n // 49) * 49 * 49 / 1e6:.2f}M → {n / 49:.0f}× fewer")

Shifted windows straddle the old window borders, so after two layers the ★ token has exchanged information with tokens from 4 different original windows. The cost grows **linearly** with the number of tokens instead of quadratically.

### 3.8 · Squeeze-and-Excitation: attention over channels
**Squeeze:** global-average-pool each channel to one number. **Excite:** a tiny 2-layer MLP turns those numbers into a weight between 0 and 1 per channel. **Scale:** multiply each channel by its weight.

In [ ]:
class SE(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc1 = nn.Conv2d(channels, channels // reduction, 1)
        self.fc2 = nn.Conv2d(channels // reduction, channels, 1)

    def forward(self, x):
        s = x.mean((2, 3), keepdim=True)                       # squeeze: (B, C, 1, 1)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(s))))       # excite: a weight per channel, per image
        return x * s                                           # scale

se = SE(64)
feat = torch.randn(2, 64, 16, 16)
print("in/out shapes:", tuple(feat.shape), tuple(se(feat).shape), "| extra parameters:", sum(p.numel() for p in se.parameters()))

---
# 🚀 4 · Benchmark: three ways to compute the same attention

- **naive:** your `attention()`, which stores the full $N \times N$ weights
- **chunked:** your online-softmax version (block 512)
- **PyTorch SDPA:** `F.scaled_dot_product_attention`, which picks a fused kernel for your hardware (FlashAttention or memory-efficient attention on NVIDIA; a fused Metal kernel on Apple GPUs)

Settings: batch 1, 8 heads, 64 dims per head.

**Memory:** on NVIDIA, `torch.cuda.max_memory_allocated` measures the true peak, so the Kaggle run reports **measured** extra memory. Apple's GPU has no peak-memory counter (its driver grabs memory in 1 GB chunks), so on a Mac the table shows the **computed** size of the biggest temporary each method creates: the score matrix and the softmax output for naive, one block of them for chunked. SDPA's fused kernel keeps its blocks in on-chip memory, so it has no such temporary.

In [ ]:
def timed(fn, reps=5):
    fn(); sync()
    t0 = time.perf_counter()
    for _ in range(reps): fn()
    sync()
    return (time.perf_counter() - t0) / reps * 1000

def peak_extra_mb(fn):
    """Measured peak extra memory of fn() on NVIDIA; NaN elsewhere (no peak counter)."""
    if device != "cuda":
        return float("nan")
    sync(); torch.cuda.empty_cache(); base = torch.cuda.memory_allocated(); torch.cuda.reset_peak_memory_stats()
    fn(); sync()
    return (torch.cuda.max_memory_allocated() - base) / 2**20

def biggest_temp_mb(name, N, heads=8, block=512):
    """Computed size of the largest temporary: scores + softmax weights (float32)."""
    return {"naive": 2 * heads * N * N * 4, "chunked": 2 * heads * N * min(block, N) * 4, "PyTorch SDPA": 0}[name] / 2**20

Ns = [256, 1024, 2048, 4096, 8192, 16384] if FULL else [256, 1024, 2048, 4096]
impls = {
    "naive": lambda q, k, v: attention(q, k, v)[0],
    "chunked": lambda q, k, v: chunked_attention(q, k, v, block=512),
    "PyTorch SDPA": lambda q, k, v: F.scaled_dot_product_attention(q, k, v),
}
bench = []
with torch.no_grad():
    for N in Ns:
        q, k, v = (torch.randn(1, 8, N, 64, device=device) for _ in range(3))
        for name, fn in impls.items():
            try:
                mem = peak_extra_mb(lambda: fn(q, k, v))
                ms = timed(lambda: fn(q, k, v), reps=3 if N >= 4096 else 10)
            except torch.OutOfMemoryError:
                ms, mem = float("nan"), float("nan"); print(f"N={N} {name}: out of memory")
            bench.append(dict(N=N, impl=name, ms=ms, measured_MB=mem, computed_MB=biggest_temp_mb(name, N)))
        del q, k, v
        if device == "cuda": torch.cuda.empty_cache()
        elif device == "mps": torch.mps.empty_cache()
bench = pd.DataFrame(bench)
mem_col = "measured_MB" if device == "cuda" else "computed_MB"
print(bench.pivot(index="N", columns="impl", values=["ms", mem_col]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for name, g in bench.groupby("impl"):
    axes[0].plot(g.N, g.ms, "o-", label=name)
    if name != "PyTorch SDPA" or device == "cuda":
        axes[1].plot(g.N, g[mem_col], "o-", label=name)
for a, t in zip(axes, ["time (ms)", f"extra memory (MB, {mem_col.split('_')[0]})"]):
    a.set_xscale("log", base=2); a.set_yscale("log"); a.set_xlabel("tokens N"); a.set_title(t); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

How to read it (measured on an M3 MacBook; your numbers will differ a bit):
- **Naive time grows ~4× per doubling of N** once N is large (11 → 42 ms from 2,048 to 4,096 tokens), and so does its memory: 1 GB of temporaries at 4,096 tokens, for one layer of one image.
- **Chunked** memory grows only linearly, but in pure PyTorch it is ~2× *slower* than naive: the Python loop launches many small GPU operations. The real win needs a fused kernel that keeps the blocks in on-chip memory.
- **PyTorch SDPA** is that fused kernel: ~2× faster than naive on the Mac (19 vs. 42 ms at 4,096 tokens), with no N×N buffer. On Kaggle's NVIDIA GPU the measured memory column shows the difference directly, and naive runs out of memory at the longest lengths while SDPA doesn't.

**Practical rule:** never write `softmax(q @ k.T) @ v` in production. Use `F.scaled_dot_product_attention` (or a library that calls it, like timm or Hugging Face with `attn_implementation="sdpa"`).

---
# 📊 5 · Evaluate: do the implementations agree?
Speed means nothing if the answers differ. Compare each implementation with a float64 reference, in float32 and float16.

In [ ]:
torch.manual_seed(0)
q, k, v = (torch.randn(1, 8, 1024, 64, dtype=torch.float64) for _ in range(3))
truth = attention(q, k, v)[0]
rows = []
for dtype in (torch.float32, torch.float16):
    qd, kd, vd = (t.to(dtype).to(device) for t in (q, k, v))     # Apple GPUs have no float64: convert on the CPU first
    for name, fn in impls.items():
        with torch.no_grad():
            err = (fn(qd, kd, vd).float().cpu().double() - truth).abs()
        rows.append(dict(dtype=str(dtype).split(".")[1], impl=name, max_abs_err=err.max().item(), mean_abs_err=err.mean().item()))
print(pd.DataFrame(rows).pivot(index="impl", columns="dtype", values=["max_abs_err", "mean_abs_err"]).to_string(float_format="%.1e"))

All three agree to better than 1e-6 in float32. In float16 the worst errors are ~3–7e-4, normal for 16-bit numbers (about 3 significant digits) and harmless for inference. SDPA is the most accurate in float16 because its kernel accumulates the sums in float32 internally. Module 15 checks whole-model accuracy after reducing precision.

---
# 🧪 6 · Experiments

### Experiment A · Do positions matter?
A toy task that **needs** order: each sequence of 16 tokens contains a token **A** once and a token **B** once (the rest is random filler). Label: *does A come before B?* A tiny 2-layer transformer averages its outputs to classify. Three versions: **no positions**, **learned position embeddings**, **RoPE** (applied inside attention).

**Predict first:** what test accuracy will each version reach? Which learns fastest?

In [ ]:
from cvlib.transformer import Block

L_TRAIN, VOCAB = 16, 10
def make_order_task(n, length, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(2, VOCAB, (n, length), generator=g)
    pos = torch.rand(n, length, generator=g).argsort(1)[:, :2]       # two distinct random positions
    x[torch.arange(n), pos[:, 0]] = 0                                 # token A
    x[torch.arange(n), pos[:, 1]] = 1                                 # token B
    return x, (pos[:, 0] < pos[:, 1]).long()

class RopeBlock(Block):
    """Same block, but rotate queries and keys (RoPE) before attention."""
    def forward(self, x):
        a, h = self.attn, self.norm1(x)
        B, N, D = h.shape
        q, k, v = (a._split(t) for t in a.qkv(h).chunk(3, dim=-1))
        o, w = attention(rope(q), rope(k), v)
        a.attn = w.detach()
        x = x + a.proj(o.transpose(1, 2).reshape(B, N, D))
        return x + self.mlp(self.norm2(x))

class OrderModel(nn.Module):
    def __init__(self, positions, dim=64, depth=2, heads=4):
        super().__init__()
        self.positions = positions
        self.emb = nn.Embedding(VOCAB, dim)
        self.pos = nn.Parameter(torch.randn(1, L_TRAIN, dim) * 0.02)
        self.blocks = nn.Sequential(*[(RopeBlock if positions == "rope" else Block)(dim, heads) for _ in range(depth)])
        self.head = nn.Linear(dim, 2)

    def forward(self, x):
        h = self.emb(x)
        if self.positions == "learned":
            h = h + self.pos[:, :x.shape[1]]
        return self.head(self.blocks(h).mean(1))

xtr, ytr = make_order_task(20000, L_TRAIN, seed=0)
xte, yte = make_order_task(2000, L_TRAIN, seed=1)
order_models, curves = {}, {}
for positions in ["none", "learned", "rope"]:
    torch.manual_seed(0)
    m = OrderModel(positions)                       # tiny model: the CPU is fastest here
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    curves[positions] = []
    t0 = time.perf_counter()
    for epoch in range(5):
        for idx in torch.randperm(len(xtr)).split(256):
            loss = F.cross_entropy(m(xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            curves[positions].append((m(xte).argmax(1) == yte).float().mean().item())
    order_models[positions] = m
    print(f"{positions:8s} test accuracy per epoch: {np.round(curves[positions], 3)} | {time.perf_counter() - t0:.0f}s")

- **No positions: 50%**, a coin flip, however long you train. The model sees a *bag* of tokens; "A before B" and "B before A" are the same bag. Nothing to learn.
- **Learned positions** get to 100%, after an epoch of figuring out what the position vectors should mean.
- **RoPE** is typically right from the first epoch: comparing *relative* positions is exactly what the task needs, and RoPE builds that into every attention score.

Let's look at what the RoPE model's attention does on one example.

In [ ]:
m = order_models["rope"]
xs, ys = xte[:1], yte[:1]
with torch.no_grad(): m(xs)
w = m.blocks[0].attn.attn[0]                                    # (heads, 16, 16)
labels = ["A" if t == 0 else "B" if t == 1 else "·" for t in xs[0].tolist()]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for h, ax in enumerate(axes):
    ax.imshow(w[h], cmap="viridis"); ax.set_title(f"layer 1, head {h}")
    ax.set_xticks(range(16)); ax.set_xticklabels(labels, fontsize=7); ax.set_yticks(range(16)); ax.set_yticklabels(labels, fontsize=7)
plt.suptitle(f"rows = queries, columns = keys | label: A before B = {bool(ys.item())}"); plt.tight_layout(); plt.show()

Look for columns that light up at the **A** and **B** positions: heads that learned "find the special tokens". The exact patterns change with the random seed.

### Experiment B · Longer inputs than in training
Real models meet longer inputs than they were trained on (bigger images, longer documents). Test the RoPE model, trained only on length 16, on sequences of length 32, 64 and 128. (The learned-embedding model can't even run: it only has 16 position vectors.)

**Predict:** does accuracy hold up?

In [ ]:
for length in [16, 32, 64, 128]:
    xl, yl = make_order_task(2000, length, seed=2)
    with torch.no_grad():
        acc = (order_models["rope"](xl).argmax(1) == yl).float().mean().item()
    print(f"RoPE model, length {length:>3}: accuracy {acc:.3f}")

Measured: ~99% at length 32 (2× the training length), but only ~65% at 64 and 128. RoPE has no table of positions that can run out, so it *runs* at any length, and it extends a little. But in training it never saw offsets above 15, and at 4× the length most A–B pairs are further apart than anything it learned a rule for.

Takeaway: "RoPE generalizes to longer inputs" is true only up to a point. That's why VLMs that accept high-resolution images use RoPE variants with scaling tricks *and* train on long inputs.

### Experiment C · Does an SE block help a small CNN?
Module 03's CNN on 10,000 CIFAR-10 images, with and without an SE block after each stage, same training recipe (5 epochs).

**Predict:** how much accuracy does SE add, and for how many extra parameters?

In [ ]:
def make_cnn(use_se, widths=(32, 64, 128)):
    layers, c = [], 3
    for wd in widths:
        layers += [conv_bn(c, wd), conv_bn(wd, wd)]
        if use_se: layers.append(SE(wd))
        layers.append(nn.MaxPool2d(2)); c = wd
    return nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))

Xc, yc = cifar10("train"); Xct, yct = cifar10("test")
idx = np.random.default_rng(0).choice(len(yc), 10000, replace=False)
results_se = []
for seed in (0, 1):
    for use_se in (False, True):
        torch.manual_seed(seed)
        tr = GPULoader(Xc[idx], yc[idx], batch_size=128, shuffle=True, augment=random_crop_flip, mean=CIFAR_MEAN, std=CIFAR_STD, device=device, seed=seed)
        te = GPULoader(Xct, yct, batch_size=500, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
        model = make_cnn(use_se)
        opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.05)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=5 * len(tr))
        t0 = time.perf_counter()
        h = cvtrain.fit(model, tr, te, nn.CrossEntropyLoss(), opt, epochs=5, device=device, batch_scheduler=sched, verbose=False)
        results_se.append(dict(seed=seed, SE=use_se, test_acc=h["val_acc"][-1], params=sum(p.numel() for p in model.parameters()),
                               seconds=time.perf_counter() - t0))
        print(results_se[-1])
df_se = pd.DataFrame(results_se)
print(df_se.groupby("SE")[["test_acc", "params", "seconds"]].mean().round(4))

Measured: **no gain**. Average accuracy was 67.1% without SE and 66.9% with it, a difference smaller than the gap between two seeds of the same model. SE added 4% more parameters and made training ~27% slower (many small operations).

That's a real result, not a bug: SE's gains in the papers (~1–2% on ImageNet) come from deep networks trained for many epochs, where there are many channels to choose between. A 6-layer net trained for 5 epochs on 10k images has nothing to gain. **Lesson:** check whether a published trick helps *your* model and data before keeping it, and compare against seed-to-seed noise (module 14 turns this into a proper significance test).

---
# 📝 7 · Recap

- **Attention** = softmax(Q·Kᵀ/√d)·V: a soft lookup. Every token mixes in information from every other token, weighted by query–key similarity.
- **√d** keeps the scores at unit variance so the softmax doesn't become one-hot.
- **Multi-head**: several smaller attentions in parallel; fused into one qkv projection in code.
- **Self vs. cross**: same sequence vs. queries from one sequence and keys/values from another (DETR, VLMs).
- Attention is **permutation-equivariant**: without positions it sees a bag of tokens. Learned and sinusoidal encodings are added to tokens; **RoPE** rotates q and k so scores depend on relative offsets.
- Cost is **O(N²)**: FlashAttention keeps it exact but avoids storing N² scores (online softmax); windowed attention restricts who attends to whom (linear cost, approximate).
- In practice use **`F.scaled_dot_product_attention`**.
- **SE blocks** are attention over channels, used inside CNNs.

### Interview questions
1. Write scaled dot-product attention. Why the √d?
2. What's the difference between self-attention and cross-attention? Give a vision example of each.
3. Why do transformers need positional encodings? What happens without them?
4. How does RoPE work, and why does it generalize to longer sequences better than learned position embeddings?
5. What's the time and memory complexity of attention? What does FlashAttention change, and what doesn't it change?
6. How does windowed attention reduce cost, and what's the purpose of shifting the windows?
7. What does a Squeeze-and-Excitation block do? Where is it used?
8. Multi-head attention with 8 heads of 64 dims vs. 1 head of 512 dims: same parameters? What's different?

**Next:** 08b turns this into a full Vision Transformer, loads pretrained weights into your own code, and compares ViTs with CNNs.